# C2 — Mixed Preprocessing Plan

---

## 🔍 Problem-এ কী চাওয়া হয়েছে?

**C-Data-1** এবং **C-Data-2** দুটো dataset একসাথে ব্যবহার করে তিনটি কাজ করতে হবে:

| Task | কী করতে হবে |
|---|---|
| **(a)** | প্রতিটি column **Nominal** নাকি **Ordinal** — চিহ্নিত করা |
| **(b)** | **Encoding plan**: কোন column → One-Hot, কোন column → Ordinal Encoding |
| **(c)** | **Scaling plan**: কোন column → Min-Max, কোন column → Standardization, কোন column → Robust |

### দুটো Dataset-এর সব Column:

**C-Data-1:** `Age`, `Hours_Study`, `GPA`, `Internet`, `City`

**C-Data-2:** `Income_BDT`, `Transactions`, `Temp_C`, `Education`, `Satisfaction`


---

## 🎯 এই কাজ থেকে আমরা কী অর্জন করতে পারব?

- দুটো dataset একসাথে দেখে **সব ধরনের column** (numeric, binary, nominal, ordinal) চিনতে পারব।
- **কোন column-এ কী করতে হবে না** — সেটাও বুঝব (binary column-কে encode বা scale করতে হয় না)।
- Real ML project-এ **end-to-end preprocessing plan** তৈরি করার দক্ষতা তৈরি হবে।


---

## 🧠 কীভাবে চিন্তা করতে হবে?

### Column দেখে সিদ্ধান্ত নেওয়ার flow:

```
Column দেখলে প্রথমে জিজ্ঞেস করো:

1. Numeric?
   → হ্যাঁ  → Scaling দরকার → outlier আছে? → Robust / Min-Max / Std
   → না   → Categorical → ক্রম আছে? → Ordinal / One-Hot

2. Binary (Yes/No, True/False)?
   → সরাসরি 0/1 map করো — encoding বা scaling দরকার নেই
```

### প্রতিটি Column-এর সিদ্ধান্ত (মাথায় রেখে কাজ করব):

| Dataset | Column | Type | সিদ্ধান্ত |
|---|---|---|---|
| C-Data-1 | `Age` | Numeric | Scaling |
| C-Data-1 | `Hours_Study` | Numeric | Scaling |
| C-Data-1 | `GPA` | Numeric | Scaling |
| C-Data-1 | `Internet` | Binary (Yes/No) | 0/1 map |
| C-Data-1 | `City` | **Nominal** | One-Hot Encoding |
| C-Data-2 | `Income_BDT` | Numeric | Scaling (outlier আছে) |
| C-Data-2 | `Transactions` | Numeric | Scaling (outlier আছে) |
| C-Data-2 | `Temp_C` | Numeric | Scaling (bounded) |
| C-Data-2 | `Education` | **Ordinal** | Ordinal Encoding |
| C-Data-2 | `Satisfaction` | **Ordinal** | Ordinal Encoding |


---

## 🛠️ Problem Solve করার Approach

**Step 1:** দুটো dataset তৈরি করা।

**Step 2 (Task a):** প্রতিটি column Nominal/Ordinal/Numeric/Binary চিহ্নিত করা।

**Step 3 (Task b):** Encoding plan implement করা।

**Step 4 (Task c):** Scaling plan implement করা।

**Step 5:** Final preprocessed dataset দেখানো।


## Step 1: উভয় Dataset তৈরি করা

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import (OrdinalEncoder, MinMaxScaler,
                                   RobustScaler, StandardScaler)

# ── Creating C-Data-1 ──
data1 = pd.DataFrame({
    'ID':           [1, 2, 3, 4, 5],
    'Age':          [20, 21, 22, 20, 23],
    'Hours_Study':  [1.0, 0.5, 2.2, 5.0, 0.2],
    'GPA':          [3.10, 2.60, 3.40, 3.90, 2.30],
    'Internet':     ['Yes', 'No', 'Yes', 'Yes', 'No'],
    'City':         ['Dhaka', 'Chattogram', 'Rajshahi', 'Dhaka', 'Rajshahi']
})

# ── Creating C-Data-2 ──
data2 = pd.DataFrame({
    'ID':           [1, 2, 3, 4, 5],
    'Income_BDT':   [30000, 45000, 52000, 300000, 38000],
    'Transactions': [0, 1, 2, 12, 0],
    'Temp_C':       [25.0, 26.0, 24.5, 28.0, 25.5],
    'Education':    ['High School', 'Bachelor', 'Master', 'Bachelor', 'Master'],
    'Satisfaction': ['Low', 'Medium', 'High', 'Medium', 'Medium']
})

print("C-Data-1:")
print(data1.to_string(index=False))
print()
print("C-Data-2:")
print(data2.to_string(index=False))


C-Data-1:
 ID  Age  Hours_Study  GPA Internet       City
  1   20          1.0  3.1      Yes      Dhaka
  2   21          0.5  2.6       No Chattogram
  3   22          2.2  3.4      Yes   Rajshahi
  4   20          5.0  3.9      Yes      Dhaka
  5   23          0.2  2.3       No   Rajshahi

C-Data-2:
 ID  Income_BDT  Transactions  Temp_C   Education Satisfaction
  1       30000             0    25.0 High School          Low
  2       45000             1    26.0    Bachelor       Medium
  3       52000             2    24.5      Master         High
  4      300000            12    28.0    Bachelor       Medium
  5       38000             0    25.5      Master       Medium


দুটো DataFrame তৈরি করা হয়েছে।
C-Data-1-এ `Internet` binary (Yes/No) এবং `City` nominal।
C-Data-2-এ `Income_BDT` ও `Transactions`-এ outlier আছে, `Temp_C` bounded।


---

## Task (a): Nominal vs Ordinal Column চিহ্নিত করা


In [2]:
# ── Building the column classification table ──
classification = pd.DataFrame({
    'Dataset':  ['C-Data-1'] * 5 + ['C-Data-2'] * 5,
    'Column':   ['Age', 'Hours_Study', 'GPA', 'Internet', 'City',
                 'Income_BDT', 'Transactions', 'Temp_C', 'Education', 'Satisfaction'],
    'Type':     ['Numeric', 'Numeric', 'Numeric', 'Binary', 'Nominal',
                 'Numeric', 'Numeric', 'Numeric', 'Ordinal', 'Ordinal'],
    'Reason':   [
        'Continuous number — age',
        'Continuous number — study hours',
        'Continuous number — grade point',
        'Yes/No — only two values, no order needed',
        'City names — no natural order',
        'Continuous number — salary',
        'Count — number of transactions',
        'Temperature reading — bounded 24.5-28',
        'HS < Bachelor < Master — clear order',
        'Low < Medium < High — clear order'
    ]
})

print("── Task (a): Column Type Classification ──")
print(classification.to_string(index=False))


── Task (a): Column Type Classification ──
 Dataset       Column    Type                                    Reason
C-Data-1          Age Numeric                   Continuous number — age
C-Data-1  Hours_Study Numeric           Continuous number — study hours
C-Data-1          GPA Numeric           Continuous number — grade point
C-Data-1     Internet  Binary Yes/No — only two values, no order needed
C-Data-1         City Nominal             City names — no natural order
C-Data-2   Income_BDT Numeric                Continuous number — salary
C-Data-2 Transactions Numeric            Count — number of transactions
C-Data-2       Temp_C Numeric     Temperature reading — bounded 24.5-28
C-Data-2    Education Ordinal      HS < Bachelor < Master — clear order
C-Data-2 Satisfaction Ordinal         Low < Medium < High — clear order


Table-এ সব ১০টি column-এর type ও কারণ একসাথে দেখা যাচ্ছে।
- `Nominal` → One-Hot Encoding লাগবে
- `Ordinal` → Ordinal Encoding লাগবে
- `Binary` → সরাসরি 0/1 map করব
- `Numeric` → Scaling লাগবে


---

## Task (b): Encoding Plan

### Encoding সিদ্ধান্ত:

| Column | Encoding | কারণ |
|---|---|---|
| `Internet` | Binary map (Yes=1, No=0) | দুটো মাত্র value — OHE-তে একটি column-ই হবে, তাই সরাসরি map |
| `City` | **One-Hot Encoding** | Nominal — Dhaka, Chattogram, Rajshahi-তে কোনো ক্রম নেই |
| `Education` | **Ordinal Encoding** | Ordinal — High School < Bachelor < Master |
| `Satisfaction` | **Ordinal Encoding** | Ordinal — Low < Medium < High |


In [3]:
# ── Step 1: Binary encoding for Internet ──
data1['Internet_Enc'] = data1['Internet'].map({'Yes': 1, 'No': 0})
# Here we've converted Yes→1 and No→0 using a simple dictionary map.

print("Internet encoded:")
print(data1[['Internet', 'Internet_Enc']].to_string(index=False))


Internet encoded:
Internet  Internet_Enc
     Yes             1
      No             0
     Yes             1
     Yes             1
      No             0


`dict.map()` দিয়ে Yes → 1, No → 0 সরাসরি convert।
এটি sklearn encoder ছাড়াই করা যায় কারণ মাত্র দুটো value।


In [4]:
# ── Step 2: One-Hot Encoding for City (Nominal) ──
city_ohe = pd.get_dummies(data1['City'], prefix='City').astype(int)
# Here we've created binary columns for each unique city.

print("City One-Hot Encoded:")
print(pd.concat([data1[['City']], city_ohe], axis=1).to_string(index=False))


City One-Hot Encoded:
      City  City_Chattogram  City_Dhaka  City_Rajshahi
     Dhaka                0           1              0
Chattogram                1           0              0
  Rajshahi                0           0              1
     Dhaka                0           1              0
  Rajshahi                0           0              1


`pd.get_dummies()` → প্রতিটি unique city-র জন্য একটি binary column।
`prefix='City'` → নতুন column-এর নাম `City_Dhaka`, `City_Chattogram`, `City_Rajshahi`।


In [5]:
# ── Step 3: Ordinal Encoding for Education ──
edu_enc = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
data2['Education_Enc'] = edu_enc.fit_transform(data2[['Education']]).astype(int)
# Here we've mapped: High School=0, Bachelor=1, Master=2

# ── Step 4: Ordinal Encoding for Satisfaction ──
sat_enc = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
data2['Satisfaction_Enc'] = sat_enc.fit_transform(data2[['Satisfaction']]).astype(int)
# Here we've mapped: Low=0, Medium=1, High=2

print("Ordinal Encoded:")
print(data2[['Education', 'Education_Enc',
             'Satisfaction', 'Satisfaction_Enc']].to_string(index=False))


Ordinal Encoded:
  Education  Education_Enc Satisfaction  Satisfaction_Enc
High School              0          Low                 0
   Bachelor              1       Medium                 1
     Master              2         High                 2
   Bachelor              1       Medium                 1
     Master              2       Medium                 1


`OrdinalEncoder(categories=[...])` → ভেতরের list-এ ক্রম অনুযায়ী category সাজানো।
`fit_transform()` → data দেখে শেখে, তারপর transform করে।
`.astype(int)` → output default float, integer-এ convert করা হয়েছে।


---

## Task (c): Scaling Plan

### Scaling সিদ্ধান্ত:

| Column | Dataset | Scaler | কারণ |
|---|---|---|---|
| `Age` | C-Data-1 | **Min-Max** | 20–23 bounded, কোনো outlier নেই |
| `Hours_Study` | C-Data-1 | **Robust** | 0.2–5.0, কিছুটা skewed — Robust নিরাপদ |
| `GPA` | C-Data-1 | **Min-Max** | 2.3–3.9 bounded, কোনো outlier নেই |
| `Income_BDT` | C-Data-2 | **Robust** | 300000 outlier আছে |
| `Transactions` | C-Data-2 | **Robust** | 12 outlier আছে |
| `Temp_C` | C-Data-2 | **Min-Max** | 24.5–28.0 bounded, কোনো outlier নেই |


In [6]:
# ── Scaling C-Data-1 numeric columns ──

# Now we'll apply Min-Max to Age and GPA (bounded, no outliers)
mm = MinMaxScaler()
data1['Age_Scaled'] = mm.fit_transform(data1[['Age']]).flatten()
data1['GPA_Scaled'] = mm.fit_transform(data1[['GPA']]).flatten()
# Here we've scaled Age and GPA to [0, 1] range.

# Now we'll apply Robust to Hours_Study (slightly skewed distribution)
rb = RobustScaler()
data1['Hours_Scaled'] = rb.fit_transform(data1[['Hours_Study']]).flatten()
# Here we've used Robust scaler for Hours_Study to handle skew safely.

print("C-Data-1 — Scaled Numeric Columns:")
print(data1[['Age', 'Age_Scaled',
             'GPA', 'GPA_Scaled',
             'Hours_Study', 'Hours_Scaled']].round(4).to_string(index=False))


C-Data-1 — Scaled Numeric Columns:
 Age  Age_Scaled  GPA  GPA_Scaled  Hours_Study  Hours_Scaled
  20      0.0000  3.1      0.5000          1.0        0.0000
  21      0.3333  2.6      0.1875          0.5       -0.2941
  22      0.6667  3.4      0.6875          2.2        0.7059
  20      0.0000  3.9      1.0000          5.0        2.3529
  23      1.0000  2.3      0.0000          0.2       -0.4706


`MinMaxScaler()` → Age ও GPA-কে [0, 1]-এ আনা হয়েছে।
`RobustScaler()` → Hours_Study-তে 5.0 কিছুটা outlier-ish, তাই Robust নিরাপদ।
`.flatten()` → 2D output-কে 1D column-এ রূপান্তর করা।


In [7]:
# ── Scaling C-Data-2 numeric columns ──

# Now we'll apply Robust to Income_BDT and Transactions (have outliers)
data2['Income_Scaled'] = rb.fit_transform(data2[['Income_BDT']]).flatten()
data2['Trans_Scaled']  = rb.fit_transform(data2[['Transactions']]).flatten()
# Here we've used Robust scaler for both — 300000 and 12 are clear outliers.

# Now we'll apply Min-Max to Temp_C (bounded, no outliers)
data2['Temp_Scaled'] = mm.fit_transform(data2[['Temp_C']]).flatten()
# Here we've scaled Temp_C to [0, 1] — perfectly bounded between 24.5 and 28.

print("C-Data-2 — Scaled Numeric Columns:")
print(data2[['Income_BDT', 'Income_Scaled',
             'Transactions', 'Trans_Scaled',
             'Temp_C', 'Temp_Scaled']].round(4).to_string(index=False))


C-Data-2 — Scaled Numeric Columns:
 Income_BDT  Income_Scaled  Transactions  Trans_Scaled  Temp_C  Temp_Scaled
      30000        -1.0714             0          -0.5    25.0       0.1429
      45000         0.0000             1           0.0    26.0       0.4286
      52000         0.5000             2           0.5    24.5       0.0000
     300000        18.2143            12           5.5    28.0       1.0000
      38000        -0.5000             0          -0.5    25.5       0.2857


`Income_BDT` ও `Transactions` উভয়েই outlier — Robust Scaler ব্যবহার করা হয়েছে।
`Temp_C` bounded range — Min-Max Scaler [0, 1]-এ সুন্দরভাবে এনেছে।


## Step 5: Final Preprocessed Dataset

In [8]:
# ── Assembling the final preprocessed C-Data-1 ──
final_data1 = pd.concat([
    data1[['Age_Scaled', 'Hours_Scaled', 'GPA_Scaled', 'Internet_Enc']],
    city_ohe
], axis=1)

print("Final Preprocessed C-Data-1:")
print(final_data1.round(4).to_string(index=False))


Final Preprocessed C-Data-1:
 Age_Scaled  Hours_Scaled  GPA_Scaled  Internet_Enc  City_Chattogram  City_Dhaka  City_Rajshahi
     0.0000        0.0000      0.5000             1                0           1              0
     0.3333       -0.2941      0.1875             0                1           0              0
     0.6667        0.7059      0.6875             1                0           0              1
     0.0000        2.3529      1.0000             1                0           1              0
     1.0000       -0.4706      0.0000             0                0           0              1


`pd.concat(..., axis=1)` → সব encoded ও scaled column পাশাপাশি জোড়া লাগানো।
C-Data-1-এ এখন কোনো text নেই — সম্পূর্ণ numeric।


In [9]:
# ── Assembling the final preprocessed C-Data-2 ──
final_data2 = data2[['Income_Scaled', 'Trans_Scaled', 'Temp_Scaled',
                      'Education_Enc', 'Satisfaction_Enc']]

print("Final Preprocessed C-Data-2:")
print(final_data2.round(4).to_string(index=False))


Final Preprocessed C-Data-2:
 Income_Scaled  Trans_Scaled  Temp_Scaled  Education_Enc  Satisfaction_Enc
       -1.0714          -0.5       0.1429              0                 0
        0.0000           0.0       0.4286              1                 1
        0.5000           0.5       0.0000              2                 2
       18.2143           5.5       1.0000              1                 1
       -0.5000          -0.5       0.2857              2                 1


C-Data-2-এও এখন কোনো text নেই।
- Numeric columns → scaled
- Ordinal columns → 0, 1, 2 এ encoded

উভয় dataset ML model-এ সরাসরি ব্যবহারের জন্য প্রস্তুত।


In [10]:
# ── Final summary table of all decisions ──
summary = pd.DataFrame({
    'Dataset':  ['C-Data-1']*5 + ['C-Data-2']*5,
    'Column':   ['Age', 'Hours_Study', 'GPA', 'Internet', 'City',
                 'Income_BDT', 'Transactions', 'Temp_C', 'Education', 'Satisfaction'],
    'Action':   ['Min-Max Scale', 'Robust Scale', 'Min-Max Scale',
                 'Binary Map (0/1)', 'One-Hot Encode',
                 'Robust Scale', 'Robust Scale', 'Min-Max Scale',
                 'Ordinal Encode', 'Ordinal Encode'],
    'Reason':   ['Bounded 20-23, no outlier',
                 'Skewed, Robust safer',
                 'Bounded 2.3-3.9, no outlier',
                 'Binary — direct map',
                 'Nominal — no order',
                 'Outlier: 300000',
                 'Outlier: 12',
                 'Bounded 24.5-28, no outlier',
                 'HS < Bachelor < Master',
                 'Low < Medium < High']
})

print("── Complete Preprocessing Plan ──")
print(summary.to_string(index=False))


── Complete Preprocessing Plan ──
 Dataset       Column           Action                      Reason
C-Data-1          Age    Min-Max Scale   Bounded 20-23, no outlier
C-Data-1  Hours_Study     Robust Scale        Skewed, Robust safer
C-Data-1          GPA    Min-Max Scale Bounded 2.3-3.9, no outlier
C-Data-1     Internet Binary Map (0/1)         Binary — direct map
C-Data-1         City   One-Hot Encode          Nominal — no order
C-Data-2   Income_BDT     Robust Scale             Outlier: 300000
C-Data-2 Transactions     Robust Scale                 Outlier: 12
C-Data-2       Temp_C    Min-Max Scale Bounded 24.5-28, no outlier
C-Data-2    Education   Ordinal Encode      HS < Bachelor < Master
C-Data-2 Satisfaction   Ordinal Encode         Low < Medium < High


এই table-ই হলো C2-এর **complete mixed preprocessing plan**।
একটি real ML project-এ data পাওয়ার পরে এই ধরনের plan তৈরি করেই কাজ শুরু করতে হয়।


---